# Degrade synthetic scans with Stable Diffusion img2img (GPU)

Runs `ml/synthetic/diffusion/degrade_scans.py` on Colab: SD 1.5 img2img in fp16 at
strengths 0.2–0.45 with fixed seeds, then the OCR validity filter. Outputs and manifests
go to Google Drive.

**Runtime → Change runtime type → T4 GPU.** About 2–3 s per variant on a T4 at 30 steps.

**Licence:** the default checkpoint, `stable-diffusion-v1-5/stable-diffusion-v1-5`, is
CreativeML OpenRAIL-M. That allows this use (synthetic test data); check the licence of
any other checkpoint before using it with `--model`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/ledgerlens/degraded"

In [ ]:
!git clone --depth 1 https://github.com/Sam15b/LedgerLens.git /content/LedgerLens
%cd /content/LedgerLens
!apt-get -qq install -y tesseract-ocr > /dev/null
# Colab ships CUDA torch; diffusers + accelerate run the pipeline, the extra adds jinja2
# and albumentations, pyarrow reads CORD.
!pip install -q -e ".[synthetic]" diffusers accelerate pyarrow
import torch

print(torch.__version__, "CUDA:", torch.cuda.is_available())

## Clean pages

Build the ledger and render the page images. This needs no GAN checkpoint (the seed rows
are resampled). To use the Day 21 GAN instead, copy its `final.pt` to Drive and pass
`--checkpoint`.

In [ ]:
!python -m ml.synthetic.seed --out data/synthetic/seed
!python -m ml.synthetic.build --invoices 5000
!python -m ml.synthetic.render.render_documents --limit 300 --types invoice

## img2img

Prompts alternate per page ("old photocopied invoice, coffee stain, faded toner" /
"crumpled paper scan, low contrast"); every page is run at every strength (default 0.2, 0.3, 0.4, 0.45) with a
fixed seed, so the output is reproducible.

In [ ]:
LIMIT = 200
PAGES = "data/synthetic/pages/invoice"
!python -m ml.synthetic.diffusion.degrade_scans run {PAGES} --out {OUT}/sd --limit {LIMIT}

## Validity filter

OCRs each variant with Tesseract and keeps it only if the invoice number, PO number and
total are still readable exactly. Also records CER against the clean PDF, for the
strength → CER curve.

In [ ]:
!python -m ml.synthetic.diffusion.degrade_scans filter {OUT}/sd/manifest.jsonl --root data/synthetic

In [ ]:
import json
from collections import defaultdict
from pathlib import Path

rows = [json.loads(line) for line in Path(OUT, "sd", "validity.jsonl").read_text().splitlines()]
by_strength = defaultdict(list)
for row in rows:
    by_strength[row["strength"]].append(row)
for strength, group in sorted(by_strength.items()):
    kept = sum(r["kept"] for r in group) / len(group)
    cer = sum(r["cer"] for r in group) / len(group)
    print(f"strength {strength:.2f}: kept {kept:.0%}, mean CER {cer:.3f} ({len(group)} pages)")